# 03. 자동 Stage 1 연구 경로 (선택)

`01`이 만든 레거시 데이터셋을 대상으로, 이 저장소의 선택적 자동 연구 루프를 실행합니다.

```text
diagnosis -> bounded proposal -> AIDM -> evidence verification -> human review
```

이 노트북은 이 저장소의 스킬 러너 `.agents/scripts/run-research-loop.sh`(=`research-orchestrator` 스킬)를 직접 호출합니다. 진단, 하나의 제한된 JSON 제안, AIDM 실험, 증적 검증까지 자동으로 수행한 뒤 **`ready_for_human_review`에서 멈춥니다.** AIDD 호출, 코드 생성, 고객 시스템 변경, 병합, 배포는 하지 않습니다.

- 먼저 `01_legacy_baseline.ipynb`를 실행해 데이터셋을 만들어 두세요.
- 출력은 `.agents/runs/notebook-03-auto/output/`에 남습니다.
- 이 루프는 게이트 임계값을 바꾸지 않고, 고객 행·타깃·비밀값을 기록하지 않습니다.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

def repo_root(start):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("power-forecasting 저장소 안에서 실행하세요.")


REPO_ROOT = repo_root(Path.cwd())
SCRIPTS = REPO_ROOT / ".agents" / "scripts"


def runner_env():
    # GUI로 띄운 Jupyter 커널은 uv/venv가 PATH에 없을 수 있어, 러너가 찾도록 보강합니다.
    env = dict(os.environ)
    extra = [str(Path(sys.executable).parent)]
    uv = shutil.which("uv") or str(Path.home() / ".local" / "bin" / "uv")
    if Path(uv).exists():
        extra.append(str(Path(uv).parent))
    env["PATH"] = os.pathsep.join(extra + [env.get("PATH", "")])
    return env


RUNNER_ENV = runner_env()
DATASET = REPO_ROOT / "artifacts" / "demo" / "dataset.csv"
assert DATASET.exists(), "먼저 01_legacy_baseline.ipynb 를 실행해 artifacts/demo/dataset.csv 를 만들어 주세요."

CONFIG_DIR = REPO_ROOT / ".agents" / "runs" / "notebook-03-auto"
CONFIG = CONFIG_DIR / "research-config.json"
OUTPUT = CONFIG_DIR / "output"
shutil.rmtree(OUTPUT, ignore_errors=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

# 01의 데이터셋을 대상으로 Stage 1 연구 루프를 구성합니다. 경로는 설정 파일 위치 기준 상대 경로입니다.
config = {
    "schema_version": "1",
    "run_id": "notebook-03-auto",
    "dataset_path": "../../../artifacts/demo/dataset.csv",
    "legacy_manifest_path": "../../fixtures/promoted-manifest.json",
    "run_dir": "output",
    "profiles": ["safe_weather"],
    "max_iterations": 1,
    "fold_count": 5,
    "objective": "NMAE",
    "minimum_improvement": 0.0,
    "max_plant_regression": 1.0,
}
CONFIG.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")

# research-orchestrator 스킬 러너: 진단 -> 제안 -> AIDM -> 검증 상태 머신을 안전하게 실행합니다.
loop = subprocess.run(
    [str(SCRIPTS / "run-research-loop.sh"), "--config", str(CONFIG)],
    cwd=REPO_ROOT, text=True, capture_output=True, env=RUNNER_ENV,
)
if loop.returncode != 0:
    raise RuntimeError(f"run-research-loop.sh 실패 (exit {loop.returncode}):\n{loop.stderr}")
print("연구 루프 실행 완료")
print(sorted(path.name for path in OUTPUT.iterdir()))

In [ ]:
diagnosis = json.loads((OUTPUT / "diagnosis.json").read_text(encoding="utf-8"))
summary = json.loads((OUTPUT / "research-summary.json").read_text(encoding="utf-8"))

# research-diagnostic: 집계 전용 데이터 품질·누수 진단 (원시 행은 남기지 않음).
print("[진단] 누수 검사:")
for check, passed in diagnosis["leakage_checks"].items():
    print(f"  - {check}: {passed}")
print(f"[진단] 추천 프로필: {diagnosis['recommended_profiles']}")

# research-verification: AIDM 증적을 게이트 변경 없이 재검증한 결과.
print(f"\n[요약] 상태: {summary['status']}")
print(f"[요약] 반복 횟수: {summary['iterations']}")
print(f"[요약] 사용한 프로필: {summary['used_profiles']}")
print(f"[요약] 검증 결과: {summary['verifier']['outcome']}")

## 사람 검토 경계

상태가 `ready_for_human_review`에서 멈췄습니다. `research-summary.json`은 연구 진단·제안·검증 결과일 뿐 release 또는 deploy 승인 증거가 **아닙니다.**

사람은 이 요약과 전체 증적(SHA-256으로 연결된 `diagnosis.json`, iteration별 `promotion_manifest.json`·`verification.json`)을 검토한 뒤, `02` 노트북의 수동 AIDD 및 `release-gate` 절차를 **별도로** 실행해야 합니다.

| 단계 | 스킬 | 반드시 멈추는 경계 |
| --- | --- | --- |
| 상태 머신 조정 | `research-orchestrator` | AIDD·코드 생성·고객 변경·병합·배포 전에 종료 |
| 집계 진단 | `research-diagnostic` | 후보를 발명하거나 AIDM·AIDD를 실행하지 않음 |
| 제한된 제안 생성 | `research-proposal` | 코드·임의 탐색·게이트 변경을 만들지 않음 |
| 증적 재검증 | `research-verification` | AIDM 재실행·release 승인·AIDD 호출을 하지 않음 |

즉, 자동 경로는 사람이 검토할 **후보와 증적을 준비**할 뿐, 개선을 확정하거나 배포하지 않습니다.